In [1]:
from pathlib import Path
import os
import time
import random
import torch

import numpy as np
import pandas as pd

from training import train_single
from data_helpers import make_loader
from models_transformer import SingleOutTransformerNet
from metrics import eval_single_metrics_test

In [2]:
NUM_LAYERS = 20
SEED = 123
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cpu")
print("Device:", DEVICE)

Device: cpu


In [3]:
OUTDIR = Path("./Results")
OUTDIR.mkdir(parents=True, exist_ok=True)
MODELSDIR = Path(f"./trained_models/seed_{SEED}/{NUM_LAYERS}layers")
HISTDIR = Path(OUTDIR / f"./train_history/seed_{SEED}/{NUM_LAYERS}layers")
METDIR = Path(OUTDIR / f"./test_metrics/seed_{SEED}/{NUM_LAYERS}layers")

MODELSDIR.mkdir(parents=True, exist_ok=True)
HISTDIR.mkdir(parents=True, exist_ok=True)
METDIR.mkdir(parents=True, exist_ok=True)

#### Load data

In [4]:
train_data = np.load("data_processed/train_data.npz")
val_data = np.load("data_processed/val_data.npz")
test_data = np.load("data_processed/test_data.npz")

X_train = train_data["x"]
y_train = train_data["age"]

X_val = val_data["x"]
y_val = val_data["age"]

X_test = test_data["x"]
y_test = test_data["age"]

In [5]:
train_data_s = np.load("data_processed/train_data_scaled.npz")
val_data_s = np.load("data_processed/val_data_scaled.npz")
test_data_s = np.load("data_processed/test_data_scaled.npz")

X_train_s = train_data_s["x"]
y_train_s = train_data_s["age"]

X_val_s = val_data_s["x"]
y_val_s = val_data_s["age"]

X_test_s = test_data_s["x"]
y_test_s = test_data_s["age"]

In [6]:
scalers = np.load("data_processed/scalers.npz")

x_mean, x_std = scalers["x_mean"], scalers["x_std"]
y_mean, y_std = scalers["age_mean"], scalers["age_std"]

In [7]:
BATCH_SIZE = 128

train_loader = make_loader(X_train_s, y_train_s, BATCH_SIZE)
val_loader = make_loader(X_val_s, y_val_s, BATCH_SIZE)
test_loader = make_loader(X_test_s, y_test_s, BATCH_SIZE)

### Models and training

In [8]:
EPOCHS = 150
# METRICS_EVERY = int(EPOCHS/8)
METRICS_EVERY = 1
LR = 1e-3
WD = 1e-5

In [9]:
IN_DIM = X_train_s.shape[1]

EMB_DIM = 64
NHEAD = 4
# NUM_LAYERS = 11
FF_DIM = 128
DROPOUT = 0.1

In [10]:
tag = "single_TR_age"

torch.manual_seed(SEED)    
model_single = SingleOutTransformerNet(IN_DIM, emb_dim=EMB_DIM, nhead=NHEAD, 
                                       num_layers=NUM_LAYERS, ff_dim=FF_DIM, 
                                       dropout=DROPOUT).to(DEVICE)    

loss = torch.nn.SmoothL1Loss()
optimizer = torch.optim.Adam(model_single.parameters(), lr=LR, weight_decay=WD)

hist_df = train_single(model_single, loss, optimizer, train_loader, val_loader, 
                       device=DEVICE, epochs=EPOCHS, metrics_every=METRICS_EVERY)

hist_df.to_csv(HISTDIR / f"train_history_{NUM_LAYERS}layers.csv", index=False)
torch.save(model_single.state_dict(), MODELSDIR / f"TR_model_{NUM_LAYERS}layers.pt")

ep=001 tr_loss=0.3764 | va_loss=0.2332 | tr R2=0.516 | va R2=0.477 | tr RMSE=0.696 | va RMSE=0.707 | tr MAE=0.555 | va MAE=0.563 | lr=1.00e-03 | bad_epochs=0
ep=002 tr_loss=0.1943 | va_loss=0.1865 | tr R2=0.627 | va R2=0.589 | tr RMSE=0.610 | va RMSE=0.627 | tr MAE=0.476 | va MAE=0.489 | lr=1.00e-03 | bad_epochs=0
ep=003 tr_loss=0.1826 | va_loss=0.1811 | tr R2=0.654 | va R2=0.599 | tr RMSE=0.589 | va RMSE=0.619 | tr MAE=0.459 | va MAE=0.481 | lr=1.00e-03 | bad_epochs=0
ep=004 tr_loss=0.1743 | va_loss=0.1754 | tr R2=0.663 | va R2=0.615 | tr RMSE=0.580 | va RMSE=0.607 | tr MAE=0.451 | va MAE=0.472 | lr=1.00e-03 | bad_epochs=0
ep=005 tr_loss=0.1660 | va_loss=0.1722 | tr R2=0.675 | va R2=0.623 | tr RMSE=0.570 | va RMSE=0.601 | tr MAE=0.444 | va MAE=0.468 | lr=1.00e-03 | bad_epochs=0
ep=006 tr_loss=0.1644 | va_loss=0.1709 | tr R2=0.670 | va R2=0.625 | tr RMSE=0.574 | va RMSE=0.599 | tr MAE=0.447 | va MAE=0.466 | lr=1.00e-03 | bad_epochs=0
ep=007 tr_loss=0.1655 | va_loss=0.1777 | tr R2=0.659

In [11]:
total_time = np.sum(hist_df["time"].to_numpy())
print(f"Total training time: {total_time} sec.")

Total training time: 2493.6653423309326 sec.


In [12]:
m_te = eval_single_metrics_test(model_single, X_test_s, y_test, y_mean, y_std, DEVICE)

single_test_rows = []
single_test_rows.append(["age", m_te["R2"], m_te["RMSE"], m_te["MAE"]])

In [13]:
single_metrics_df = pd.DataFrame(
    single_test_rows,
    columns=["output", "R2", "RMSE", "MAE"]
)

single_metrics_df.to_csv(METDIR / f"test_metrics_{NUM_LAYERS}layers.csv", 
                         index=False, float_format="%.6f")

In [14]:
single_metrics_df

,output,R2,RMSE,MAE
0,age,0.686432,7.637055,5.959639
